# SynapseForge — Mediclaim Agentic Workflow & Neural Tool Routing Tester

This notebook provides an end-to-end testing playground for the **Mediclaim Processing** scenario. It demonstrates how **three specialized AI agents** collaborate to process a medical insurance claim:

1. **Claim Processing Agent (Orchestrator)**: Manages end-to-end claim registration, requests details, coordinates policy verification, runs billing validation, calculates claimable amounts, and submits the claim.
2. **Policy Agent**: Insurance policy verification specialist. Fetches policy details (`POL-999`) and checks treatment-specific coverage limits.
3. **Billing Agent**: Hospital billing analyst. Fetches discharge summaries and verifies itemized medical bills for a patient.

### Dynamic Neural Tool Routing
Instead of statically binding all available tools to the LLM context, which degrades accuracy and increases token usage, this workflow leverages the **`NeuralToolRouter`** (via `RouterService`).
* Before the LLM runs at each step, the `RouterService` performs a semantic similarity search using `pgvector` to identify the **top-k** most relevant tools for the current task.
* Only these selected tools (plus any assigned collaborator agents) are bound to the LLM during execution.

## 1. Environment Setup & Imports

First, we'll locate the backend root, load the `.env` configuration file, and import the required database engines, models, and execution frameworks.

In [1]:
import os
import sys
import json
import uuid
import asyncio
from pathlib import Path

# 1. Add the backend directory to sys.path
backend_path = None
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "db" / "models.py").exists() and (p / "services" / "langgraph_dynamic_agent_executor.py").exists():
        backend_path = p
        break

if not backend_path:
    backend_path = Path.cwd() / "backend" if (Path.cwd() / "backend").exists() else Path.cwd()

sys.path.insert(0, str(backend_path))
print(f"✓ Backend directory added to path: {backend_path}")

# 2. Load environment variables from backend .env file
from dotenv import load_dotenv
dotenv_path = backend_path / ".env"
load_dotenv(dotenv_path=dotenv_path)
print(f"✓ Loaded environment variables from: {dotenv_path}")

# 3. Import database and agent executors
import db.engine
from db.models import Agent, Tool, LLMConfig, Workspace
from services.langgraph_dynamic_agent_executor import DynamicLangGraphAgentExecutor
from services.router_service import RouterService

✓ Backend directory added to path: /Users/gurvindersingh/Documents/development/repositories/personal/synapse-forge/backend
✓ Loaded environment variables from: /Users/gurvindersingh/Documents/development/repositories/personal/synapse-forge/backend/.env


/Users/gurvindersingh/Documents/development/repositories/personal/synapse-forge/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Initialize Database & Inspect Agent Configuration

We connect to the PostgreSQL database and inspect the existing agent configurations to verify that our Claim Processing, Policy, and Billing agents have been correctly seeded.

In [2]:
# Initialize database connection
await db.engine.init_db()

async with db.engine._session_factory() as session:
    from sqlalchemy import select
    res = await session.execute(select(Agent))
    agents = res.scalars().all()
    
    print("=" * 80)
    print("ACTIVE AGENTS IN DATABASE")
    print("=" * 80)
    for a in agents:
        if a.name in ["Claim Processing Agent", "Policy Agent", "Billing Agent"]:
            a.max_iterations = 3
            print(f"• Name: {a.name}")
            print(f"  ID: {a.id}")
            print(f"  Workspace ID: {a.workspace_id}")
            print(f"  Use Neural Router: {a.use_neural_router}")
            print(f"  Router Top-K: {a.router_top_k}")
            print(f"  Max Iterations: {a.max_iterations}")
            print(f"  Collaborators IDs: {a.collaborator_agent_ids}")
            print("-" * 50)

ACTIVE AGENTS IN DATABASE
• Name: Claim Processing Agent
  ID: 8e27c031-7752-4ef2-b7bb-909982356d61
  Workspace ID: 00000000-0000-0000-0000-000000000001
  Use Neural Router: True
  Router Top-K: 3
  Max Iterations: 3
  Collaborators IDs: [UUID('eb9b84cb-3512-46e3-aab5-89e73dd8c9e3'), UUID('c2187bcf-7b28-478f-9aab-9b762c805218')]
--------------------------------------------------
• Name: Policy Agent
  ID: eb9b84cb-3512-46e3-aab5-89e73dd8c9e3
  Workspace ID: 00000000-0000-0000-0000-000000000001
  Use Neural Router: True
  Router Top-K: 3
  Max Iterations: 3
  Collaborators IDs: None
--------------------------------------------------
• Name: Claim Processing Agent
  ID: 15be7d3f-ab0a-47ac-9d2c-b25cbf93e27b
  Workspace ID: 00000000-0000-0000-0000-000000000002
  Use Neural Router: True
  Router Top-K: 3
  Max Iterations: 3
  Collaborators IDs: [UUID('d82efc6e-ff32-4ac9-aac4-208ed46464a5'), UUID('5b35861f-24b8-45fb-9a33-beb3f46ec262')]
--------------------------------------------------
• Na

## 3. Streaming and Logging Configuration

Configure whether to display the token-by-token reasoning streams and intermediate logs (thoughts, tools, neural router details) during execution.

In [3]:
# Streaming & verbosity toggles
ENABLE_STREAMING = False           # Toggle agent token-by-token reasoning text stream
ENABLE_INTERMEDIATE_LOGS = False   # Toggle thoughts, tool calls/results, and neural routing detail logs

## 4. SSE Event Formatter for Rich Visualization

The execution engine streams various event types (e.g. `thought`, `router`, `tool_call`, `tool_result`, `assistant`, `error`, `complete`).
To make it clean and readable, we'll define a function `format_and_print_event` that parses these server-sent events, prints relevant fields, and applies colors and indentation based on delegation nesting depth.

In [4]:
def format_and_print_event(event_str: str) -> None:
    if not event_str.startswith("data:"):
        return
    
    data_str = event_str.removeprefix("data:").strip()
    if not data_str:
        return
        
    try:
        data = json.loads(data_str)
        event_type = data.get("type")
        label = data.get("label", "")
        detail = data.get("detail", "")
        status = data.get("status", "")
        metadata = data.get("metadata", {})
        
        # Read streaming configuration flags dynamically from globals
        show_streaming = globals().get("ENABLE_STREAMING", True)
        show_intermediate = globals().get("ENABLE_INTERMEDIATE_LOGS", True)
        
        # Color coding for terminal output
        GREEN = "\033[92m"
        BLUE = "\033[94m"
        CYAN = "\033[96m"
        YELLOW = "\033[93m"
        RED = "\033[91m"
        MAGENTA = "\033[95m"
        BOLD = "\033[1m"
        RESET = "\033[0m"
        
        # Indentation based on depth for collaborator tracing
        depth = metadata.get("depth", 0) if isinstance(metadata, dict) else 0
        indent = "  " * depth
        
        # Format based on event type
        if event_type == "reasoning":
            if not show_streaming:
                return
            if len(detail) < 15:
                sys.stdout.write(detail)
                sys.stdout.flush()
            else:
                print(f"\n{indent}💭 {BOLD}{CYAN}[REASONING]{RESET} {label}")
                print(f"{indent}   {detail}")
                
        elif event_type == "router":
            if not show_intermediate:
                return
            print(f"\n{indent}🔍 {BOLD}{YELLOW}[NEURAL ROUTER]{RESET} {label}")
            selected_tools = metadata.get("selected_tools", [])
            strategy = metadata.get("strategy", "unknown")
            print(f"{indent}   Strategy: {strategy} | Tools Selected: {[t['name'] for t in selected_tools]}")
            
        elif event_type == "thought":
            if not show_intermediate:
                return
            print(f"\n{indent}🧠 {BOLD}{BLUE}[THOUGHT]{RESET} {label}")
            if detail and len(detail.strip()) > 0:
                print(f"{indent}   {detail.strip()}")
                
        elif event_type == "tool_call":
            if not show_intermediate:
                return
            print(f"\n{indent}🔧 {BOLD}{MAGENTA}[TOOL CALL]{RESET} {label}")
            if detail:
                try:
                    tool_args = json.loads(detail)
                    print(f"{indent}   Arguments: {json.dumps(tool_args.get('arguments', {}), indent=2)}")
                except:
                    print(f"{indent}   {detail}")
                    
        elif event_type == "tool_result":
            if not show_intermediate:
                return
            print(f"\n{indent}✅ {BOLD}{GREEN}[TOOL RESULT]{RESET} {label}")
            if detail:
                truncated = detail[:400] + "..." if len(detail) > 400 else detail
                print(f"{indent}   Output: {truncated}")
                
        elif event_type == "collaborator":
            if not show_intermediate:
                return
            print(f"\n{indent}🤝 {BOLD}{CYAN}[COLLABORATOR HANDOFF]{RESET} {label}")
            if detail:
                print(f"{indent}   {detail}")
                
        elif event_type == "assistant":
            print(f"\n\n{indent}🤖 {BOLD}{GREEN}[ASSISTANT RESPONSE - {metadata.get('agent_name', 'Agent')}]{RESET}")
            print(f"{indent}{'='*80}")
            print(f"{indent}{detail}")
            print(f"{indent}{'='*80}\n")
            
        elif event_type == "error":
            print(f"\n{indent}❌ {BOLD}{RED}[ERROR]{RESET} {label}: {detail}")
            
        elif event_type == "complete":
            print(f"\n{indent}🏁 {BOLD}{GREEN}[COMPLETE]{RESET} {label}")
            
    except Exception as e:
        print(f"\nError parsing event: {event_str} -> {e}")

## 5. Run the End-to-End Mediclaim Processing Scenario

We define a function `run_scenario` which instantiates the `DynamicLangGraphAgentExecutor` and runs our test query.

We use the **Workspace 2 (Demo Workspace)** by default:
- Workspace ID: `00000000-0000-0000-0000-000000000002`
- Claim Processing Agent ID: `15be7d3f-ab0a-47ac-9d2c-b25cbf93e27b`

In [5]:
async def run_scenario(workspace_uuid_str: str, agent_uuid_str: str, prompt: str):
    workspace_id = uuid.UUID(workspace_uuid_str)
    agent_id = uuid.UUID(agent_uuid_str)
    
    print(f"🚀 Starting agent execution loop...")
    print(f"   Workspace: {workspace_id}")
    print(f"   Agent ID: {agent_id}")
    print(f"   Prompt: '{prompt}'\n")
    
    async with db.engine._session_factory() as session:
        agent = await session.get(Agent, agent_id)
        if not agent:
            print(f"❌ Error: Agent with ID {agent_id} not found in database.")
            return
            
        executor = DynamicLangGraphAgentExecutor(session)
        
        async for event in executor.execute_agent(
            agent=agent,
            user_prompt=prompt,
            depth=0,
            router_top_k_override=3
        ):
            format_and_print_event(event)

# Workspace 2 configuration details
demo_workspace_id = "00000000-0000-0000-0000-000000000002"
demo_agent_id = "15be7d3f-ab0a-47ac-9d2c-b25cbf93e27b"
test_prompt = "Process mediclaim for patient 1024 with policy POL-999 for knee replacement surgery."

await run_scenario(demo_workspace_id, demo_agent_id, test_prompt)

🚀 Starting agent execution loop...
   Workspace: 00000000-0000-0000-0000-000000000002
   Agent ID: 15be7d3f-ab0a-47ac-9d2c-b25cbf93e27b
   Prompt: 'Process mediclaim for patient 1024 with policy POL-999 for knee replacement surgery.'



Creating dynamic LangGraph agent with model: >> ollama/granite4.1:8b




Router query: >> Process mediclaim for patient 1024 with policy POL-999 for knee replacement surgery.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17398.14it/s]




Router metadata: >> {'strategy': 'neural_router', 'router_top_k': 3, 'query': 'Process mediclaim for patient 1024 with policy POL-999 for knee replacement surgery.', 'selected_tool_ids': ['d7420830-a6cb-4774-b0d8-9b8c7d5cb2ba', 'a71ec0d8-1810-45b7-adf3-533de57702b5', 'eb519614-56ec-44d3-a884-721bc816c95c'], 'selected_agent_ids': ['d82efc6e-ff32-4ac9-aac4-208ed46464a5', '5b35861f-24b8-45fb-9a33-beb3f46ec262'], 'ranked_tools': [{'id': 'd7420830-a6cb-4774-b0d8-9b8c7d5cb2ba', 'name': 'submit_mediclaim', 'type': 'MCP_TOOL', 'score': 0.4842, 'description': 'Submit the final mediclaim for processing.', 'parameters': {'type': 'object', 'required': ['policy_number', 'patient_id', 'claim_amount'], 'properties': {'patient_id': {'type': 'string', 'description': 'The patient ID (e.g., "1024")'}, 'claim_amount': {'type': 'number', 'description': 'The final claimable amount'}, 'policy_number': {'type': 'string', 'description': 'The policy number (e.g., "POL-999")'}}, 'additionalProperties': False}}

INFO:tool_router.mcp_client:Connecting to MCP server: 922b9c29-54f9-46d5-860a-b7ef699481d3
INFO:tool_router.mcp_client:Replaced 'python' command with sys.executable: /Users/gurvindersingh/Documents/development/repositories/personal/synapse-forge/.venv/bin/python
INFO:tool_router.mcp_client:Resolved relative path '../examples/beeai_mediclaim_processing/mock_fastmcp_server.py' to absolute path '/Users/gurvindersingh/Documents/development/repositories/personal/synapse-forge/examples/beeai_mediclaim_processing/mock_fastmcp_server.py'
INFO:tool_router.mcp_client:✓ Connected to stdio server 922b9c29-54f9-46d5-860a-b7ef699481d3
INFO:tool_router.mcp_client:Listed 6 tools from 922b9c29-54f9-46d5-860a-b7ef699481d3
INFO:tool_router.mcp_client:Calling tool: 922b9c29-54f9-46d5-860a-b7ef699481d3.check_coverage_limits with args: {'policy_number': 'POL-999', 'treatment_type': 'knee_replacement'}
INFO:tool_router.mcp_client:✓ Tool 922b9c29-54f9-46d5-860a-b7ef699481d3.check_coverage_limits executed succ



  🤖 [ASSISTANT RESPONSE - Policy Agent]
  {  
  "name": "check_coverage_limits",  
  "arguments": {  
    "policy_number": "POL-999",  
    "treatment_type": "knee_replacement"  
  }  
}



...



Router query: >> Submit mediclaim for patient ID 1024 with policy POL-999 using calculated claimable amount.


Batches: 100%|██████████| 1/1 [00:00<00:00, 96.24it/s]
INFO:ntr.router:Cache MISS for workspace=00000000-0000-0000-0000-000000000002 → 10 tools  (21.9 ms)
12:10:05 - LiteLLM:INFO: utils.py:4054 - 
LiteLLM completion() model= granite4.1:8b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= granite4.1:8b; provider = ollama




Router metadata: >> {'strategy': 'neural_router', 'router_top_k': 3, 'query': 'Submit mediclaim for patient ID 1024 with policy POL-999 using calculated claimable amount.', 'selected_tool_ids': ['d7420830-a6cb-4774-b0d8-9b8c7d5cb2ba', 'e765c9c4-625b-4882-abc4-f5a4d1e725e3', 'eb519614-56ec-44d3-a884-721bc816c95c'], 'selected_agent_ids': ['d82efc6e-ff32-4ac9-aac4-208ed46464a5', '5b35861f-24b8-45fb-9a33-beb3f46ec262'], 'ranked_tools': [{'id': 'd7420830-a6cb-4774-b0d8-9b8c7d5cb2ba', 'name': 'submit_mediclaim', 'type': 'MCP_TOOL', 'score': 0.5529, 'description': 'Submit the final mediclaim for processing.', 'parameters': {'type': 'object', 'required': ['policy_number', 'patient_id', 'claim_amount'], 'properties': {'patient_id': {'type': 'string', 'description': 'The patient ID (e.g., "1024")'}, 'claim_amount': {'type': 'number', 'description': 'The final claimable amount'}, 'policy_number': {'type': 'string', 'description': 'The policy number (e.g., "POL-999")'}}, 'additionalProperties': 

INFO:ntr.services.dynamic_langgraph_executor:Successfully parsed 1 tool call(s) from content: ['submit_mediclaim']




🤖 [ASSISTANT RESPONSE - Claim Processing Agent]
{  
  "name": "delegate_to_policy_agent",  
  "arguments": {  
    "task": "Verify coverage for policy POL-999 and treatment type knee_replacement, including limits and co-pay percentage."  
  }  
}



## 6. Interactive Testing Playground

Try executing different prompts here to test individual features, like asking about a policy directly or requesting billing details, to see how the Neural Router responds.

In [ ]:
# Execute a custom prompt
custom_prompt = "What are the coverage limits under policy POL-999?"
await run_scenario(demo_workspace_id, demo_agent_id, custom_prompt)